In [24]:
import sys
import json
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from metab_processing.LinearRegression import signature_helpers as sh, signature_batch as sb
from metab_processing.LinearRegression.signature_configs import CONFIGS

cfg = CONFIGS['cervical']

# Which ratio plots `view()` renders (keys of sh.plot_res).
PLOTS = ('variance_ratio',)
# ---- Which cached fit to READ (must match a computed params subdir) ----
# Default = this config's fit (l1 @ penalty 0.005). To read a different fit, override, e.g.:
#   READ = dict(method='ridge', penalty=800)
# READ = dict(method=cfg.method, penalty=cfg.penalty, l1_ratio=cfg.l1_ratio)
READ = dict(method='l1', penalty=.005, l1_ratio=cfg.l1_ratio)

## Factor-overlap tools — `compare_tcell_factors(sig)`

Compare which neighborhood factors are **selected** (subsample interval excludes 0) across a
target's T-cell runs. For one signature it prints:

1. the **symmetric difference** of the two pooled runs — *all T cells, no batch* vs *all T
   cells, cell-type (batch) covariate* — one line each direction;
2. per single T-cell type, factors selected in **that type but in neither pooled run**;
3. per pooled run, factors selected there but in **none of the single-type runs**.

Flexible: pass `top_n=` for the top-N by |β|, override the read fit (`method=`, `penalty=`),
or build your own sets and call the core `compare_factor_sets(pooled_nobatch, pooled_batch,
per_type)` for any other definition of "in".

In [ ]:
# ======================================================================================
# Factor-overlap tools: compare which factors are SELECTED across a target's T-cell runs.
#
# "Selected" (a factor is "in" a run) = its subsample beta interval excludes 0
# (`sh.filter_nonzero_interval`, touch_zero) -- the same sign-stable test the ratio plots
# highlight. Pass `top_n` to restrict to the top-N selected by |beta|, or build your own
# sets and call the flexible core `compare_factor_sets` directly for any other definition.
#
# The three cervical T-cell run-specs this compares (see `sb.runspecs(cfg)`):
#   'all_pooled'                        -- all T cells pooled, NO covariate      (no batch)
#   f'all_pooled_cov_{cell_type_col}'   -- all T cells pooled, cell-type covariate (batch)
#   f'{ct}_only'  for each ct           -- one single T subtype at a time
# ======================================================================================

def selected_factors(res, *, touch_zero=True, top_n=None):
    """The SELECTED factors of one run's `res`: those whose subsample beta interval excludes
    0 (`sh.filter_nonzero_interval` -- the test the ratio plots highlight). With `top_n`,
    keep only the top-N of those by |beta|. Returns a set of factor names; an empty set for
    a None/empty `res` (e.g. a run not yet computed)."""
    if res is None or len(res) == 0:
        return set()
    sig = sh.filter_nonzero_interval(res, touch_zero=touch_zero)
    if top_n is not None:
        sig = sig.reindex(sig['beta'].abs().sort_values(ascending=False).index).head(top_n)
    return set(sig['factor'])


def compare_factor_sets(pooled_nobatch, pooled_batch, per_type, *,
                        nobatch_name='all T cells (no batch)',
                        batch_name='all T cells (batch cov)'):
    """Flexible comparison of a target's factor sets across its runs -- PRINTS only, returns
    nothing. All three inputs are SETS OF FACTOR NAMES (e.g. from `selected_factors`), so
    this works for any 'in' definition you choose:
        pooled_nobatch -- factors in the all-T-cell run with NO covariate,
        pooled_batch   -- factors in the all-T-cell run WITH the cell-type (batch) covariate,
        per_type       -- {cell_type_name: set of factors} for each single-T-type run.
    It prints:
      (1) how the two pooled runs differ (symmetric difference, one line per direction);
      (2) per cell type, factors in THAT type but in NEITHER pooled run;
      (3) per pooled run, factors in it but in NONE of the single-type runs."""
    def _fmt(s):
        return ', '.join(sorted(s)) if s else '(none)'

    # (1) pooled no-batch vs pooled batch -- symmetric difference, two directions.
    print(f'=== pooled runs differ: {nobatch_name}  vs  {batch_name} ===')
    only_nb = pooled_nobatch - pooled_batch
    only_b = pooled_batch - pooled_nobatch
    print(f'  in {nobatch_name} but NOT {batch_name} ({len(only_nb)}): {_fmt(only_nb)}')
    print(f'  in {batch_name} but NOT {nobatch_name} ({len(only_b)}): {_fmt(only_b)}')

    # (2) per single-type: factors in that type but in NEITHER pooled run.
    pooled_union = pooled_nobatch | pooled_batch
    print(f'\n=== per T-cell type: SELECTED factors not in EITHER pooled run ===')
    for ct, fs in per_type.items():
        extra = fs - pooled_union
        print(f'  {ct} ({len(extra)}): {_fmt(extra)}')

    # (3) per pooled run: factors in it but in NO single-type run.
    per_type_union = set().union(*per_type.values()) if per_type else set()
    print(f'\n=== pooled-only: factors in a pooled run but in NO single-type run ===')
    print(f'  {nobatch_name} ({len(pooled_nobatch - per_type_union)}): '
          f'{_fmt(pooled_nobatch - per_type_union)}')
    print(f'  {batch_name} ({len(pooled_batch - per_type_union)}): '
          f'{_fmt(pooled_batch - per_type_union)}')


def compare_tcell_factors(sig, *, touch_zero=True, top_n=None, **read):
    """Read `sig`'s cervical cache and run `compare_factor_sets` over its T-cell runs:
    all_pooled (no batch), all_pooled_cov_<cell_type_col> (batch covariate), and each
    {ct}_only single-type run. 'In' = selected (subsample interval excludes 0); pass `top_n`
    for the top-N by |beta|. READ params apply; override per-call (e.g. method='ridge',
    penalty=800). Runs not yet computed are warned about and treated as empty."""
    read = {**READ, **read}
    batch_label = f'all_pooled_cov_{cfg.cell_type_col}'
    missing = []

    def sel(label):
        res = sb.load_res(cfg, sig, label, **read)
        if res is None:
            missing.append(label)
        return selected_factors(res, touch_zero=touch_zero, top_n=top_n)

    pooled_nobatch = sel('all_pooled')
    pooled_batch = sel(batch_label)
    per_type = {ct: sel(f'{ct}_only') for ct in cfg.t_cell_types}

    print(f'########## {sig}'
          + (f'  (top_n={top_n})' if top_n is not None else '') + ' ##########')
    if missing:
        print(f'  [not yet computed, treated as empty]: {missing}')
    compare_factor_sets(pooled_nobatch, pooled_batch, per_type)

In [ ]:
# Example: compare factor selection across the T-cell runs for one target.
compare_tcell_factors('HALLMARK_IL2_STAT5_SIGNALING')

# Variations:
#   compare_tcell_factors('HALLMARK_IL2_STAT5_SIGNALING', top_n=10)          # top 10 by |beta|
#   compare_tcell_factors('HALLMARK_IL2_STAT5_SIGNALING', method='ridge', penalty=800)


In [25]:
membership = json.load(open(Path(cfg.cache_dir) / '_membership.json'))
print(f'{cfg.name}: loaded membership for {len(membership)} gene set(s)')

cervical: loaded membership for 33 gene set(s)


## cervical: target signatures -- membership, missing, present, factor-removed, remaining

In [26]:
print(f'===== {cfg.name} target signatures: membership + factor-removal =====')
for name in list(cfg.target_signatures) + list(cfg.neg_controls) + list(cfg.manual_sets):
    m = membership.get(name, {})
    print(f'\n{name}')
    print(f"  present ({len(m.get('present', []))}): {m.get('present', [])}")
    print(f"  missing ({len(m.get('missing', []))}): {m.get('missing', [])}")
    print(f"  total: {m.get('total', 0)}")
    print(f"  factor_removed ({len(m.get('factor_removed', []))}): {m.get('factor_removed', [])}")
    print(f"  remaining ({len(m.get('remaining', []))}): {m.get('remaining', [])}")

===== cervical target signatures: membership + factor-removal =====

HALLMARK_IL2_STAT5_SIGNALING
  present (199): ['ABCB1', 'ADAM19', 'AGER', 'AHCY', 'AHNAK', 'AHR', 'ALCAM', 'AMACR', 'ANXA4', 'APLP1', 'ARL4A', 'BATF', 'BATF3', 'BCL2', 'BCL2L1', 'BHLHE40', 'BMP2', 'BMPR2', 'CA2', 'CAPG', 'CAPN3', 'CASP3', 'CCND2', 'CCND3', 'CCNE1', 'CCR4', 'CD44', 'CD48', 'CD79B', 'CD81', 'CD83', 'CD86', 'CDC42SE2', 'CDC6', 'CDCP1', 'CDKN1C', 'CISH', 'CKAP4', 'COCH', 'COL6A1', 'CSF1', 'CSF2', 'CST7', 'CTLA4', 'CTSZ', 'CXCL10', 'CYFIP1', 'DCPS', 'DENND5A', 'DHRS3', 'DRC1', 'ECM1', 'EEF1AKMT1', 'EMP1', 'ENO3', 'ENPP1', 'EOMES', 'ETFBKMT', 'ETV4', 'F2RL2', 'FAH', 'FGL2', 'FLT3LG', 'FURIN', 'GABARAPL1', 'GADD45B', 'GALM', 'GATA1', 'GBP4', 'GLIPR2', 'GPR65', 'GPR83', 'GPX4', 'GSTO1', 'GUCY1B1', 'HIPK2', 'HK2', 'HOPX', 'HUWE1', 'HYCC2', 'ICOS', 'IFITM3', 'IFNGR1', 'IGF1R', 'IGF2R', 'IKZF2', 'IKZF4', 'IL10', 'IL10RA', 'IL13', 'IL18R1', 'IL1R2', 'IL1RL1', 'IL2RA', 'IL2RB', 'IL3RA', 'IL4R', 'IRF4', 'IRF6', 'IR

## View helper -- `view(sig)` renders every cached run-spec for one signature

In [ ]:
TARGETS = sb.all_targets(cfg)   # targets + neg controls + manual sets (+ colon single gene)
specs = sb.runspecs(cfg)


def view(sig):
    for spec in specs:
        res = sb.load_res(cfg, sig, spec['label'], **READ)
        if res is None:
            print('not yet computed:', sig, spec['label'])
            continue
        sh.plot_res(res, highlight_genes=membership.get(sig, {}).get('present', []),
                    title=f"{sig} | {spec['label']}", plots=PLOTS)

## Metabolic targets (predicted like the T-cell signatures)

In [ ]:
view('HALLMARK_HYPOXIA')
compare_tcell_factors('HALLMARK_HYPOXIA')

In [ ]:
view('HALLMARK_GLYCOLYSIS')
compare_tcell_factors('HALLMARK_GLYCOLYSIS')

In [ ]:
view('HALLMARK_OXIDATIVE_PHOSPHORYLATION')
compare_tcell_factors('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
view('HALLMARK_FATTY_ACID_METABOLISM')
compare_tcell_factors('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
view('HALLMARK_MTORC1_SIGNALING')
compare_tcell_factors('HALLMARK_MTORC1_SIGNALING')

## Subsample stats from cached raw betas -- call `stats('SIGNATURE_NAME')`

In [33]:
# 95% CI / selection-frequency / stability, computed directly from the cached raw subsample
# betas. (CI at n_sub resamples is coarse; under L1 betas pile at 0 so selection_freq + a
# CI excluding 0 are the honest signals, p_stability is a conservative heuristic.)
from IPython.display import display

def stats(sig):
    for spec in specs:
        betas = sb.load_betas(cfg, sig, spec['label'], **READ)
        if betas is None:
            print('not yet computed:', sig, spec['label']); continue
        print(f"\n=== {sig} | {spec['label']} ===")
        display(sh.subsample_stats(betas).sort_values('selection_freq', ascending=False).head(25))

## T-cell TARGET signatures

### HALLMARK_IL2_STAT5_SIGNALING

In [ ]:
view('HALLMARK_IL2_STAT5_SIGNALING')
compare_tcell_factors('HALLMARK_IL2_STAT5_SIGNALING')

### KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY

In [ ]:
view('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')
compare_tcell_factors('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

### GOBP_T_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')
compare_tcell_factors('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

### KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')
compare_tcell_factors('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

### HALLMARK_TNFA_SIGNALING_VIA_NFKB

In [ ]:
view('HALLMARK_TNFA_SIGNALING_VIA_NFKB')
compare_tcell_factors('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

### HALLMARK_INTERFERON_GAMMA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_GAMMA_RESPONSE')
compare_tcell_factors('HALLMARK_INTERFERON_GAMMA_RESPONSE')

### HALLMARK_INTERFERON_ALPHA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_ALPHA_RESPONSE')
compare_tcell_factors('HALLMARK_INTERFERON_ALPHA_RESPONSE')

### GOBP_CELL_KILLING

In [ ]:
view('GOBP_CELL_KILLING')
compare_tcell_factors('GOBP_CELL_KILLING')

### GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY

In [ ]:
view('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')
compare_tcell_factors('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')

### GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY

In [ ]:
view('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')
compare_tcell_factors('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')

## Negative controls

### HALLMARK_PANCREAS_BETA_CELLS

In [ ]:
view('HALLMARK_PANCREAS_BETA_CELLS')
compare_tcell_factors('HALLMARK_PANCREAS_BETA_CELLS')

### HALLMARK_SPERMATOGENESIS

In [ ]:
view('HALLMARK_SPERMATOGENESIS')
compare_tcell_factors('HALLMARK_SPERMATOGENESIS')

## MANUAL_ISLET_NEG_CONTROL (manual negative control)

In [ ]:
view('MANUAL_ISLET_NEG_CONTROL')
compare_tcell_factors('MANUAL_ISLET_NEG_CONTROL')